# 005 Stock Agent Demo v1

这是第五份 OpenAI 学习 Notebook。

学习目标：

1. 把 Structured Outputs 和 Function Calling 串起来
2. 做一个最小可运行的股票助手 Demo
3. 理解“意图识别 -> 工具调用 -> 结果组织”的完整链路
4. 为后续接入 FastAPI 接口做准备

这一份不再只讲单个能力，而是把前两阶段的能力真正组合起来。

参考文档：

- Structured Outputs：https://platform.openai.com/docs/guides/structured-outputs
- Function Calling：https://platform.openai.com/docs/guides/function-calling


## 先理解这份 Demo 的边界

这个版本只做最小功能闭环：

1. 查询最新价格
2. 查询涨跌幅
3. 查询公司简介
4. 查询最近新闻摘要

刻意不做这些内容：

- 技术指标分析
- 自动交易建议
- 长篇研究报告
- 多 Agent 协作

原因很简单：第一版先追求“可用”，不要过早追求“炫技”。


## 这份 Demo 的内部流程

目标流程如下：

```text
用户问题
  -> 模型识别意图
  -> 应用选择是否触发工具调用
  -> 获取 mock 行情/新闻/公司信息
  -> 模型组织自然语言答案
  -> 返回统一结果
```

你可以把它理解成一个最小版股票智能体内核。


## 加载环境变量

这里继续沿用前几份 Notebook 的方式：自动向上查找项目根目录 `.env`。


In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv


def load_project_env() -> Path | None:
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        env_path = path / ".env"
        if env_path.exists():
            load_dotenv(env_path, override=False)
            return env_path
    return None


env_path = load_project_env()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL") or None

print(f"Loaded .env: {env_path}" if env_path else "No .env found")
print("OPENAI_MODEL =", OPENAI_MODEL)
print("OPENAI_API_KEY loaded =", bool(OPENAI_API_KEY))
print("OPENAI_BASE_URL =", OPENAI_BASE_URL)


## 创建客户端

这份 Notebook 仍然优先采用 `Chat Completions API`，因为它更容易兼容现有私有网关。


In [ ]:
from openai import OpenAI


if not OPENAI_API_KEY:
    raise ValueError("请先在项目根目录 .env 中配置 OPENAI_API_KEY")


client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_BASE_URL,
)


## 第一步：定义结构化意图对象

这一步沿用 `003` 的思路。

目标不是直接回答用户，而是先让模型把用户问题识别成稳定字段。

这就像 Java 服务里先把自然语言请求转成一个 `IntentDTO`。


In [ ]:
from typing import Literal

from pydantic import BaseModel, Field


class StockIntent(BaseModel):
    intent: Literal["stock_quote", "stock_news", "company_profile", "general_chat", "unknown"] = Field(
        description="用户意图类型"
    )
    symbol: str | None = Field(default=None, description="股票代码，例如 AAPL、NVDA、TSLA")
    needs_tool: bool = Field(description="是否需要调用外部工具")


## 第二步：定义意图识别提示词

这里要尽量把意图边界说清楚。

例如：

- 问价格、涨跌 -> `stock_quote`
- 问最近新闻 -> `stock_news`
- 问公司是什么 -> `company_profile`
- 普通闲聊 -> `general_chat`


In [ ]:
INTENT_SYSTEM_PROMPT = """
你是一个股票助手的意图识别器。

请根据用户输入识别以下字段：
1. intent:
   - stock_quote: 查询价格、走势、涨跌幅
   - stock_news: 查询最近新闻、热点、消息
   - company_profile: 查询公司简介、主营业务、行业
   - general_chat: 普通聊天或不需要调用工具的问题
   - unknown: 无法判断
2. symbol: 如果用户明确提到股票代码，则提取，例如 AAPL、NVDA、TSLA
3. needs_tool: 如果需要查询外部数据，则为 true；否则为 false

回答时请严格遵守 schema。
""".strip()


## 第三步：封装意图识别函数

这里继续使用 `parse(..., response_format=StockIntent)`。

目的很明确：先把自由文本变成程序稳定可用的结构体。


In [ ]:
def parse_stock_intent(message: str) -> StockIntent:
    completion = client.beta.chat.completions.parse(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": INTENT_SYSTEM_PROMPT},
            {"role": "user", "content": message},
        ],
        response_format=StockIntent,
    )

    choice = completion.choices[0]
    refusal = getattr(choice.message, "refusal", None)
    if refusal:
        raise ValueError(f"模型拒绝回答: {refusal}")

    parsed = choice.message.parsed
    if parsed is None:
        raise ValueError("模型没有返回可解析的意图结果")

    return parsed


## 先试一下意图识别

你可以先验证：不同自然语言，是否会被识别成不同意图。


In [ ]:
intent_examples = [
    "苹果现在股价多少？",
    "英伟达最近有什么新闻？",
    "AAPL 是什么公司？",
    "你好，给我介绍一下你自己",
]

for message in intent_examples:
    parsed = parse_stock_intent(message)
    print("user:", message)
    print("intent:", parsed.model_dump())
    print("-" * 60)


## 第四步：定义 mock 工具

这里沿用 `004` 的思路，先不接真实行情 API。

这样你能先把智能体的控制流程跑通。


In [ ]:
def get_stock_quote(symbol: str) -> dict:
    mock_data = {
        "AAPL": {"symbol": "AAPL", "price": 215.32, "change_percent": 1.82},
        "NVDA": {"symbol": "NVDA", "price": 118.47, "change_percent": -0.64},
        "TSLA": {"symbol": "TSLA", "price": 177.91, "change_percent": 2.15},
    }
    return mock_data.get(symbol.upper(), {"symbol": symbol.upper(), "price": None, "change_percent": None})


def get_stock_news(symbol: str) -> dict:
    mock_news = {
        "AAPL": [
            "苹果继续推进生成式 AI 能力整合。",
            "市场关注新一季 iPhone 销售预期。",
        ],
        "NVDA": [
            "英伟达数据中心业务仍然是市场焦点。",
            "AI 芯片供应链动态持续受关注。",
        ],
    }
    return {"symbol": symbol.upper(), "news": mock_news.get(symbol.upper(), ["暂无新闻数据"]) }


def get_company_profile(symbol: str) -> dict:
    mock_profiles = {
        "AAPL": {"symbol": "AAPL", "company_name": "Apple Inc.", "industry": "Consumer Electronics"},
        "NVDA": {"symbol": "NVDA", "company_name": "NVIDIA Corporation", "industry": "Semiconductors"},
    }
    return mock_profiles.get(symbol.upper(), {"symbol": symbol.upper(), "company_name": "Unknown", "industry": "Unknown"})


## 第五步：把意图映射到工具

这一步是这份 Demo 的核心连接点。

前面 `003` 学的是“模型提取结构化字段”，`004` 学的是“模型/应用如何调用工具”。

到了这里，你要学的是：

- 识别出什么意图
- 就路由到什么工具

这很像 Java 后端里的策略分发。


In [ ]:
TOOL_BY_INTENT = {
    "stock_quote": get_stock_quote,
    "stock_news": get_stock_news,
    "company_profile": get_company_profile,
}


## 第六步：执行工具并拿到结构化业务数据

这里我们先不用让模型自己挑工具，而是先基于前一步的意图识别结果，由应用代码直接路由。

这么做的好处是：

- 流程更容易理解
- 调试更简单
- 更适合第一版教学


In [ ]:
def execute_tool_by_intent(parsed_intent: StockIntent) -> dict | None:
    if not parsed_intent.needs_tool:
        return None

    if not parsed_intent.symbol:
        return {"error": "缺少股票代码，暂时无法调用工具"}

    tool_func = TOOL_BY_INTENT.get(parsed_intent.intent)
    if tool_func is None:
        return {"error": f"当前意图 {parsed_intent.intent} 没有对应工具"}

    return tool_func(parsed_intent.symbol)


## 先测试“意图 -> 工具结果”这一步

如果这一层跑通，说明你的业务数据获取链路已经成立。


In [ ]:
message = "英伟达最近有什么新闻？"
parsed_intent = parse_stock_intent(message)
tool_result = execute_tool_by_intent(parsed_intent)

print("parsed_intent =", parsed_intent.model_dump())
print("tool_result =", tool_result)


## 第七步：定义回答整理提示词

到这里，工具已经拿到数据了。

最后一步是让模型把：

- 用户原问题
- 识别出的意图
- 工具结果

组织成自然语言答案。

这一层的职责不是“查数据”，而是“把数据讲清楚”。


In [ ]:
ANSWER_SYSTEM_PROMPT = """
你是一个股票助手。
请根据给定的用户问题、识别出的意图和工具结果，用简洁中文生成最终回答。

要求：
1. 不要编造工具结果中不存在的数据
2. 直接回答用户问题
3. 如果工具结果中有 error，就说明原因
4. 如果没有拿到工具结果，但问题属于普通聊天，可以直接回答
""".strip()


## 第八步：封装最终回答函数

这里用普通聊天生成最终答案就够了。

因为结构化数据和工具结果前面都已经准备好了，现在模型只负责表达。


In [ ]:
import json


def render_final_answer(user_message: str, parsed_intent: StockIntent, tool_result: dict | None) -> str:
    prompt = {
        "user_message": user_message,
        "parsed_intent": parsed_intent.model_dump(),
        "tool_result": tool_result,
    }

    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": ANSWER_SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps(prompt, ensure_ascii=False)},
        ],
    )
    return response.choices[0].message.content or ""


## 第九步：封装最小股票助手

现在把整个流程串起来：

1. 先识别意图
2. 再执行工具
3. 最后生成答案

这就是一个最小可运行的业务 Agent。


In [ ]:
def stock_agent_chat(user_message: str) -> dict:
    parsed_intent = parse_stock_intent(user_message)
    tool_result = execute_tool_by_intent(parsed_intent)
    final_answer = render_final_answer(user_message, parsed_intent, tool_result)

    return {
        "user_message": user_message,
        "parsed_intent": parsed_intent.model_dump(),
        "tool_result": tool_result,
        "final_answer": final_answer,
    }


## 试几个完整问题

这一步就是阶段 5 最重要的验收。

你应该能看到：

- 不同问题会走不同意图
- 不同意图会触发不同工具
- 最终都会回到自然语言答案


In [ ]:
questions = [
    "苹果现在股价多少？",
    "英伟达最近有什么新闻？",
    "AAPL 是什么公司？",
]

for question in questions:
    result = stock_agent_chat(question)
    print("user:", result["user_message"])
    print("intent:", result["parsed_intent"])
    print("tool_result:", result["tool_result"])
    print("final_answer:", result["final_answer"])
    print("=" * 80)


## 如果用户没有提供股票代码怎么办

这是第一版很容易遇到的问题。

例如用户说：

```python
"最近有什么芯片股新闻？"
```

这时候：

- 模型可能能识别出是 `stock_news`
- 但不一定能提取出明确 `symbol`

第一版里，我们直接返回一个简单错误或澄清提示就够了。

不要一开始就把股票模糊匹配、实体识别、向量召回全加进来。


In [ ]:
result = stock_agent_chat("最近有什么芯片股新闻？")
print(result)


## 这一版为什么先不用模型自己选工具

你可能会问：`004` 已经学了 Function Calling，为什么这里又变成应用自己按意图路由？

原因是这份 `005` 的教学重点是“把能力串起来”，不是一下子把所有复杂度叠满。

当前拆法更适合初学阶段：

1. `Structured Outputs` 负责把用户问题变成稳定意图
2. 应用代码负责做明确路由
3. 最终模型负责把结果讲出来

等这条链路稳定之后，再升级到：

- 模型先识别意图
- 然后模型自己决定调用哪个工具
- 再支持多工具组合

这样会更顺。


## 和 FastAPI 接口的关系

下一步接到 FastAPI 时，你大概率会做一个接口：

- `POST /api/v1/stock-agent/chat`

请求体大致可以是：

```json
{
  "message": "苹果现在股价多少？"
}
```

响应体可以直接复用这个 Notebook 里 `stock_agent_chat()` 的结果结构。


In [ ]:
# 伪代码示例
request_body = {"message": "苹果现在股价多少？"}
response_body = stock_agent_chat(request_body["message"])
response_body


## 当前阶段结论

你现在需要记住：

1. 最小股票智能体并不复杂，本质上就是“识别 -> 路由 -> 取数 -> 组织答案”
2. Structured Outputs 很适合做第一层意图识别
3. 第一版先用应用代码显式路由，比一开始就全交给模型更容易控场
4. 只要 mock 工具链路能跑通，后面替换成真实行情 API 就不难
5. 这一版做完后，你已经具备了把能力接进 FastAPI 的基础

下一份建议学习：

- 把这个 Demo 接成真实 FastAPI 接口
- 再考虑引入标准 Agents SDK
